In [2]:
import os
from pathlib import Path
import findspark

os.environ["SPARK_HOME"] = "/workspaces/Stream_Analytics-Group_7/spark-4.0.1-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

findspark.init()

import pyspark
print("PySpark version:", pyspark.__version__)
print("SPARK_HOME:", os.environ["SPARK_HOME"])
print("JAVA_HOME:", os.environ["JAVA_HOME"])

PySpark version: 4.0.1
SPARK_HOME: /workspaces/Stream_Analytics-Group_7/spark-4.0.1-bin-hadoop3
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [3]:
import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("group7-analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 17:29:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1


In [5]:
from pathlib import Path

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

bronze_orders_path = repo_root / "data" / "bronze" / "orders"
bronze_delivery_path = repo_root / "data" / "bronze" / "delivery"

print("Repo root:", repo_root)
print("Orders bronze path:", bronze_orders_path)
print("Delivery bronze path:", bronze_delivery_path)
print("Orders exists:", bronze_orders_path.exists())
print("Delivery exists:", bronze_delivery_path.exists())

Repo root: /workspaces/Stream_Analytics-Group_7
Orders bronze path: /workspaces/Stream_Analytics-Group_7/data/bronze/orders
Delivery bronze path: /workspaces/Stream_Analytics-Group_7/data/bronze/delivery
Orders exists: True
Delivery exists: True


In [6]:
orders = spark.read.parquet(str(bronze_orders_path))
delivery = spark.read.parquet(str(bronze_delivery_path))

print("Orders:", orders.count())
print("Delivery:", delivery.count())

orders.show(5, truncate=False)
delivery.show(5, truncate=False)

Orders: 94
Delivery: 738


+------------------------------------+------------------------------------+----------------+-----------------------+-----------------------+-------------+-------+----------+-----------+--------+--------------------------+-------+-----------------------+-----------------------+
|event_id                            |order_id                            |event_type      |event_ts               |ingest_ts              |restaurant_id|zone_id|courier_id|order_value|is_promo|estimated_delivery_minutes|version|event_time             |ingest_time            |
+------------------------------------+------------------------------------+----------------+-----------------------+-----------------------+-------------+-------+----------+-----------+--------+--------------------------+-------+-----------------------+-----------------------+
|64dce1a6-2081-43d3-a08e-37e1a9b9aa4d|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|prep_started    |2026-04-16 20:19:24.093|2026-04-16 20:36:46.102|rest_20      |east   |NULL

In [7]:
from pyspark.sql.functions import (
    window,
    count,
    when,
    sum,
    col,
    coalesce,
    lit
)

In [8]:
orders_kpi = (
    orders
    .groupBy(
        window("event_time", "5 minutes"),
        "zone_id"
    )
    .agg(
        count("*").alias("orders_count")
    )
    .orderBy("window")
)

orders_kpi.show(truncate=False)

+------------------------------------------+-------+------------+
|window                                    |zone_id|orders_count|
+------------------------------------------+-------+------------+
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|west   |10          |
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|south  |6           |
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|east   |8           |
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|center |19          |
|{2026-04-16 20:20:00, 2026-04-16 20:25:00}|west   |3           |
|{2026-04-16 20:20:00, 2026-04-16 20:25:00}|center |4           |
|{2026-04-16 20:20:00, 2026-04-16 20:25:00}|east   |1           |
|{2026-04-16 20:25:00, 2026-04-16 20:30:00}|south  |3           |
|{2026-04-16 20:25:00, 2026-04-16 20:30:00}|east   |2           |
|{2026-04-16 20:30:00, 2026-04-16 20:35:00}|east   |2           |
|{2026-04-16 20:30:00, 2026-04-16 20:35:00}|center |1           |
|{2026-04-16 20:35:00, 2026-04-16 20:40:00}|center |13          |
|{2026-04-

In [9]:
cancel_kpi = (
    orders
    .withColumn("is_cancelled", when(col("event_type") == "cancelled", 1).otherwise(0))
    .groupBy(
        window("event_time", "5 minutes"),
        "zone_id"
    )
    .agg(
        sum("is_cancelled").alias("cancel_count")
    )
    .orderBy("window")
)

cancel_kpi.show(truncate=False)

+------------------------------------------+-------+------------+
|window                                    |zone_id|cancel_count|
+------------------------------------------+-------+------------+
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|west   |0           |
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|south  |0           |
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|east   |0           |
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|center |0           |
|{2026-04-16 20:20:00, 2026-04-16 20:25:00}|west   |2           |
|{2026-04-16 20:20:00, 2026-04-16 20:25:00}|center |1           |
|{2026-04-16 20:20:00, 2026-04-16 20:25:00}|east   |0           |
|{2026-04-16 20:25:00, 2026-04-16 20:30:00}|south  |0           |
|{2026-04-16 20:25:00, 2026-04-16 20:30:00}|east   |0           |
|{2026-04-16 20:30:00, 2026-04-16 20:35:00}|east   |0           |
|{2026-04-16 20:30:00, 2026-04-16 20:35:00}|center |0           |
|{2026-04-16 20:35:00, 2026-04-16 20:40:00}|center |0           |
|{2026-04-

In [10]:
delivered_kpi = (
    orders
    .withColumn("is_delivered", when(col("event_type") == "delivered", 1).otherwise(0))
    .groupBy(
        window("event_time", "5 minutes"),
        "zone_id"
    )
    .agg(
        sum("is_delivered").alias("delivered_count")
    )
    .orderBy("window")
)

delivered_kpi.show(truncate=False)

+------------------------------------------+-------+---------------+
|window                                    |zone_id|delivered_count|
+------------------------------------------+-------+---------------+
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|west   |0              |
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|south  |0              |
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|east   |0              |
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|center |0              |
|{2026-04-16 20:20:00, 2026-04-16 20:25:00}|west   |0              |
|{2026-04-16 20:20:00, 2026-04-16 20:25:00}|center |0              |
|{2026-04-16 20:20:00, 2026-04-16 20:25:00}|east   |0              |
|{2026-04-16 20:25:00, 2026-04-16 20:30:00}|south  |0              |
|{2026-04-16 20:25:00, 2026-04-16 20:30:00}|east   |0              |
|{2026-04-16 20:30:00, 2026-04-16 20:35:00}|east   |0              |
|{2026-04-16 20:30:00, 2026-04-16 20:35:00}|center |0              |
|{2026-04-16 20:35:00, 2026-04-16 

In [11]:
waiting_orders = (
    orders
    .filter(col("event_type") == "created")
    .groupBy("zone_id")
    .agg(count("*").alias("waiting_orders"))
)

waiting_orders.show()

+-------+--------------+
|zone_id|waiting_orders|
+-------+--------------+
|   east|             3|
| center|             8|
|   west|             5|
|  south|             2|
+-------+--------------+



In [12]:
available_delivery = (
    delivery
    .filter(col("event_type") == "status_change")
    .groupBy("zone_id")
    .agg(count("*").alias("available_couriers"))
)

available_delivery.show()

+-------+------------------+
|zone_id|available_couriers|
+-------+------------------+
|   east|                66|
|  north|                54|
| center|                56|
|   west|                58|
|  south|                42|
+-------+------------------+



In [13]:
demand_supply = (
    waiting_orders
    .join(available_delivery, on="zone_id", how="outer")
    .fillna(0)
)

demand_supply.show()

+-------+--------------+------------------+
|zone_id|waiting_orders|available_couriers|
+-------+--------------+------------------+
| center|             8|                56|
|   east|             3|                66|
|  north|             0|                54|
|  south|             2|                42|
|   west|             5|                58|
+-------+--------------+------------------+



In [14]:
demand_supply = demand_supply.withColumn(
    "demand_supply_ratio",
    col("waiting_orders") / (col("available_couriers") + lit(1))
)

demand_supply.show(truncate=False)

+-------+--------------+------------------+--------------------+
|zone_id|waiting_orders|available_couriers|demand_supply_ratio |
+-------+--------------+------------------+--------------------+
|center |8             |56                |0.14035087719298245 |
|east   |3             |66                |0.04477611940298507 |
|north  |0             |54                |0.0                 |
|south  |2             |42                |0.046511627906976744|
|west   |5             |58                |0.0847457627118644  |
+-------+--------------+------------------+--------------------+



In [15]:
demand_supply.orderBy(col("waiting_orders").desc()).show(truncate=False)

+-------+--------------+------------------+--------------------+
|zone_id|waiting_orders|available_couriers|demand_supply_ratio |
+-------+--------------+------------------+--------------------+
|center |8             |56                |0.14035087719298245 |
|west   |5             |58                |0.0847457627118644  |
|east   |3             |66                |0.04477611940298507 |
|south  |2             |42                |0.046511627906976744|
|north  |0             |54                |0.0                 |
+-------+--------------+------------------+--------------------+



In [16]:
demand_supply.orderBy(col("demand_supply_ratio").desc()).show(truncate=False)

+-------+--------------+------------------+--------------------+
|zone_id|waiting_orders|available_couriers|demand_supply_ratio |
+-------+--------------+------------------+--------------------+
|center |8             |56                |0.14035087719298245 |
|west   |5             |58                |0.0847457627118644  |
|south  |2             |42                |0.046511627906976744|
|east   |3             |66                |0.04477611940298507 |
|north  |0             |54                |0.0                 |
+-------+--------------+------------------+--------------------+

